# 11 - Evaluate RAG Retrieval
This notebook checks whether `search()` (from notebook 10) actually finds the right bulletin
passages for a set of test questions, in both Finnish and English.

We measure:
- **hit@k** - was a correct passage in the top k results?
- **MRR** - how high did it rank, on average?
- **NDCG** - a version of MRR that also credits multiple good results, not just one
- A manual review table for **citation precision / completeness / faithfulness**
  (these need a human to actually read the results, so there is no formula for them)


## Step 0: Set up and connect to the index

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass


Mounted at /content/drive


In [3]:
!pip install -q chromadb sentence-transformers pyyaml pandas

from pathlib import Path
import yaml
import chromadb
from chromadb.utils import embedding_functions

REPO = Path.cwd()
config = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))

client = chromadb.PersistentClient(path=str(REPO / config["vector_store"]["persist_dir"]))
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config["embedding_candidates"][0]["id"]
)
collection = client.get_collection(
    config["vector_store"]["collection_name"], embedding_function=embedding_fn
)
print(collection.count(), "chunks available to search")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

6982 chunks available to search


## Step 1: The `search()` function
Same as notebook 10 - reused here so this notebook can run on its own.


In [4]:
def search(query, k=5, before_date=None):
    where = None
    if before_date:
        where = {"published_number": {"$lte": int(before_date.replace("-", ""))}}
    result = collection.query(
        query_texts=[query], n_results=k, where=where,
        include=["documents", "metadatas", "distances"]
    )
    hits = []
    for doc, meta, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        hits.append({
            "similarity": round(1 - distance, 3),
            "title": meta.get("title"),
            "text": doc,
            "published": meta.get("published"),
            "source": meta.get("source"),
        })
    return hits


### Optional: mock mode
If the index isn't built yet, flip `MOCK_MODE = True` so the rest of this notebook still
runs end-to-end on fake data. Set it back to `False` once notebook 09 finishes.


In [5]:
MOCK_MODE = False  # real index is ready

def mock_search(query, k=5, before_date=None):
    # fake results just so the rest of the notebook can be tested right now
    return [
        {"similarity": 0.81, "title": "tem_2024_03", "text": "Avoimia tyopaikkoja oli maaliskuussa yhteensa 45 000...", "published": "2024-03-28", "source": "tem"},
        {"similarity": 0.74, "title": "tem_2024_02", "text": "Tyollisyyskatsaus helmikuu 2024...", "published": "2024-02-28", "source": "tem"},
        {"similarity": 0.60, "title": "statfin_release_1", "text": "Job vacancy survey results...", "published": None, "source": "statfin"},
    ][:k]

active_search = mock_search if MOCK_MODE else search


## Step 2: Build the test question set
Write questions BEFORE looking at any results. `expected_keyword` is something that should
appear in the title of a correct result - swap these for real keywords once you know what's
actually in your corpus (check `documents.csv` from notebook 08/09).


In [6]:
test_cases = [
    # KEHA - Finnish (diacritics fixed: a->ä, o->ö where needed)
    {"query": "Kuinka monta avointa työpaikkaa oli elokuussa 2025?", "expected_keyword": "elokuu 2025", "lang": "fi"},
    {"query": "Mitä työllisyyskatsaus kertoo kesäkuulta 2025?", "expected_keyword": "kesäkuu 2025", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli maaliskuussa 2025?", "expected_keyword": "maaliskuu 2025", "lang": "fi"},
    {"query": "Mikä oli tilanne huhtikuussa 2025?", "expected_keyword": "huhtikuu 2025", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli lokakuussa 2025?", "expected_keyword": "lokakuu 2025", "lang": "fi"},
    {"query": "Mitä tapahtui heinäkuussa 2026?", "expected_keyword": "heinäkuu 2026", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli elokuussa 2026?", "expected_keyword": "elokuu 2026", "lang": "fi"},
    {"query": "Mikä on työllisyyskatsaus?", "expected_keyword": "Työllisyyskatsaus", "lang": "fi"},

    # KEHA - Finnish, more months/years (tem source)
    {"query": "Kuinka monta työpaikkaa oli syyskuussa 2020?", "expected_keyword": "syyskuu 2020", "lang": "fi"},
    {"query": "Mikä oli tilanne lokakuussa 2022?", "expected_keyword": "lokakuu 2022", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli maaliskuussa 2021?", "expected_keyword": "maaliskuu 2021", "lang": "fi"},
    {"query": "Mitä tapahtui joulukuussa 2022?", "expected_keyword": "joulukuu 2022", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli kesäkuussa 2019?", "expected_keyword": "kesäkuu 2019", "lang": "fi"},
    {"query": "Mikä oli tilanne lokakuussa 2015?", "expected_keyword": "10/2015", "lang": "fi"},

    # KEHA - Finnish, batch 3 (new)
    {"query": "Kuinka monta työpaikkaa oli syyskuussa 2025?", "expected_keyword": "syyskuu 2025", "lang": "fi"},
    {"query": "Mikä oli tilanne marraskuussa 2024?", "expected_keyword": "marraskuu 2024", "lang": "fi"},
    {"query": "Kuinka monta työpaikkaa oli toukokuussa 2026?", "expected_keyword": "toukokuu 2026", "lang": "fi"},

    # KEHA - English
    {"query": "How many job vacancies were there in May 2026?", "expected_keyword": "May 2026", "lang": "en"},
    {"query": "What happened to vacancies in March 2026?", "expected_keyword": "March 2026", "lang": "en"},
    {"query": "How many vacancies were reported in April 2025?", "expected_keyword": "April 2025", "lang": "en"},
    {"query": "What was the employment situation in June 2025?", "expected_keyword": "June 2025", "lang": "en"},
    {"query": "How many job vacancies in July 2026?", "expected_keyword": "July 2026", "lang": "en"},
    {"query": "What is the employment bulletin for August 2026?", "expected_keyword": "August 2026", "lang": "en"},
    {"query": "What is the employment bulletin?", "expected_keyword": "Employment Bulletin", "lang": "en"},

    # KEHA - English, batch 3 (new) - note: tests the ENGLISH version of Aug 2025, contrast with the broken Finnish one above
    {"query": "What was the employment situation in October 2025?", "expected_keyword": "October 2025", "lang": "en"},
    {"query": "How many vacancies were there in January 2026?", "expected_keyword": "January 2026", "lang": "en"},
    {"query": "What happened to vacancies in August 2025?", "expected_keyword": "August 2025", "lang": "en"},

    # Statistics Finland (statfin)
    {"query": "How many job vacancies were there in the second quarter?", "expected_keyword": "second quarter of 2026", "lang": "en"},
    {"query": "Were job vacancies lower in the first quarter of 2026?", "expected_keyword": "first quarter of 2026", "lang": "en"},
    {"query": "Were there fewer job vacancies in 2025 than the year before?", "expected_keyword": "Fewer job vacancies in 2025", "lang": "en"},
]

print(len(test_cases), "test questions:", sum(c["lang"]=="fi" for c in test_cases), "fi /",
      sum(c["lang"]=="en" for c in test_cases), "en")


30 test questions: 17 fi / 13 en


## Step 3: Run search for every question and record where the expected result landed

In [7]:
def rank_of_expected(hits, expected_keyword):
    """Return the 1-based rank of the first hit whose title contains expected_keyword, or None."""
    for i, hit in enumerate(hits, start=1):
        if expected_keyword.lower() in (hit["title"] or "").lower():
            return i
    return None

K = 5
results_log = []
for case in test_cases:
    hits = active_search(case["query"], k=K)
    rank = rank_of_expected(hits, case["expected_keyword"])
    results_log.append({**case, "hits": hits, "rank": rank})

    lang = case["lang"]
    query = case["query"]
    print(f"\n[{lang}] {query}")
    print("  expected keyword:", case["expected_keyword"], "| found at rank:", rank)
    for h in hits:
        sim = h["similarity"]
        title = h["title"]
        print(f"    ({sim}) {title}")



[fi] Kuinka monta avointa työpaikkaa oli elokuussa 2025?
  expected keyword: elokuu 2025 | found at rank: None
    (0.714) Työllisyyskatsaus elokuu 2026
    (0.702) Työllisyyskatsaus, elokuu 2024
    (0.686) Työllisyyskatsaus, elokuu 2020
    (0.683) Työllisyyskatsaus, elokuu 2021
    (0.683) Työllisyyskatsaus, elokuu 2022

[fi] Mitä työllisyyskatsaus kertoo kesäkuulta 2025?
  expected keyword: kesäkuu 2025 | found at rank: None
    (0.635) Employment Bulletin December 2025 ﻿ ﻿
    (0.635) Employment Bulletin June 2025
    (0.632) Työllisyyskatsaus 9/2015
    (0.631) Työllisyyskatsaus kesäkuu 2026
    (0.627) Employment Bulletin June 2025

[fi] Kuinka monta työpaikkaa oli maaliskuussa 2025?
  expected keyword: maaliskuu 2025 | found at rank: 5
    (0.673) Työllisyyskatsaus, maaliskuu 2023
    (0.656) Työllisyyskatsaus, maaliskuu 2021
    (0.651) Työllisyyskatsaus maaliskuu 2026 ﻿ ﻿
    (0.636) Työllisyyskatsaus, maaliskuu 2022
    (0.634) Työllisyyskatsaus maaliskuu 2025

[fi] Mikä ol

## Step 4: hit@k

In [8]:
for k in [1, 3, 5]:
    hits_at_k = sum(1 for r in results_log if r["rank"] is not None and r["rank"] <= k)
    print(f"hit@{k}: {hits_at_k}/{len(results_log)} = {hits_at_k/len(results_log):.0%}")


hit@1: 23/30 = 77%
hit@3: 26/30 = 87%
hit@5: 28/30 = 93%


## Step 5: MRR (Mean Reciprocal Rank)

In [9]:
reciprocal_ranks = [1 / r["rank"] if r["rank"] else 0 for r in results_log]
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
print(f"MRR: {mrr:.3f}")


MRR: 0.824


## Step 6: NDCG (simplified, binary relevance)
Since we only have one "correct" passage per question here (not graded relevance), this is
NDCG with binary relevance - it rewards the correct passage appearing, and ranks it higher
the closer it is to position 1.


In [10]:
import math

def ndcg_at_k(rank, k):
    if rank is None or rank > k:
        return 0.0
    dcg = 1 / math.log2(rank + 1)
    idcg = 1 / math.log2(1 + 1)  # best possible: correct result at rank 1
    return dcg / idcg

ndcg_scores = [ndcg_at_k(r["rank"], K) for r in results_log]
print(f"NDCG@{K}: {sum(ndcg_scores)/len(ndcg_scores):.3f}")


NDCG@5: 0.851


## Step 7: Summary

In [12]:
summary = {
    "num_questions": len(results_log),
    "hit@1": sum(1 for r in results_log if r["rank"] == 1) / len(results_log),
    "hit@3": sum(1 for r in results_log if r["rank"] and r["rank"] <= 3) / len(results_log),
    "hit@5": sum(1 for r in results_log if r["rank"] and r["rank"] <= 5) / len(results_log),
    "mrr": mrr,
    f"ndcg@{K}": sum(ndcg_scores) / len(ndcg_scores),
}
for k, v in summary.items():
    print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")


num_questions: 30
hit@1: 0.767
hit@3: 0.867
hit@5: 0.933
mrr: 0.824
ndcg@5: 0.851


## Step 8: Findings

## Tested 30 questions (17 Finnish, 13 English) against the RAG index - 180 documents, 6,982 chunks.

**The headline numbers look strong:** hit@1 = 77%, hit@5 = 93%, MRR = 0.82.

**But reading the actual retrieved text (not just titles) told a different, more useful story.**

**English KEHA bulletins** are genuinely solid - real sentences, real numbers, correct answers every time I checked.

**Finnish KEHA bulletins** are hit-or-miss. Retrieval often finds the right month, but frequently pulls a chart/table chunk (occupation category labels and axis numbers) instead of an actual sentence with a real vacancy count. So the document is right, but the chunk isn't always usable.

**Statistics Finland (statfin) sources** have a real bug - both examples I checked came back as website navigation menu text ("Search Menu Statistics Show subpages...") instead of the actual article. This scored as a "hit" on paper but would be useless in practice.

**Bottom line:** the automatic score measures whether the right *document* was found - it doesn't measure whether the retrieved *text* is actually good enough to answer with. Those are two different things, and this evaluation shows they don't always line up. Worth someone looking at the statfin scraping and the Finnish chunking before this goes further.